### 파이프라인 설계
- 문서를 넣으면 벡터 DB 알아서 적재
- 과제로 하기

### 랭체인 체인


In [2]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


load_dotenv() #.env 파일에 저장된 api 키 로드

True

In [3]:

#벡터 저장소 설정
question = "대한민국의 청년 지원 정책 알려줘"
messages = [

    HumanMessage(content=question)
]

result = model.invoke(messages)
print(result)

NameError: name 'model' is not defined

In [10]:
# 모델 설정
model = init_chat_model("openai:gpt-5.6-luna")

# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"


#임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

load_vs = Chroma(
    collection_name="K_ladder_2026",
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

In [ ]:
## 체인흐름
# 모델 -> 프롬프트 -> 출력형식

chain = model
chain.invoke("청년 월세 지원")

In [ ]:
chain = model | StrOutputParser()
chain.invoke("청년 월세 지원")

In [ ]:
prompt = [

    HumanMessage(content="너는 정책 안내 담당자야. 다음질문에 친절하게 냥냥체로 답변해.{question}")

    chain = prompt | model | StrOutputParser()
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("너는 정책 안내 담당자야. 다음 질문에 친절하게 냥냥체 답변해라. {quesion}")



In [ ]:
chain = prompt | model | StrOutputParser()
chain

In [ ]:
prompt_test = ChatPromptTemplate.from_messages([
    ('system',"너는 {character}야. 말투도 {character}말투로 답변해")
    ("human","까칠하게 질문에 답을 하도록 해{quetion}")
])

chain = prompt_test | model | StrOutputParser()
chain.invoke({"character":"어린아이", "question":"오늘 밥에 할일 추천"})
print(result)

<>:2: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:2: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
C:\Users\Lee\AppData\Local\Temp\ipykernel_24684\1755113850.py:2: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ('system',"너는 {character}야. 말투도 {character}말투로 답변해")


NameError: name 'ChatPromptTemplate' is not defined

In [4]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List
#출력형식을 변경

In [ ]:
class AnswerFormat():
        answer : str = Field(description="질문에 해당하는 답변")
        recommend : List[str] = Field(description="내일 할일 추천")

parser = PydanticOutputParser(pydantic_object=AnswerFormat)


my_test = ChatPromptTemplate.from_messages([

        ('system', "너는 {character} 말투도 {character}답변해")
        ('humna, "친절히 답변',question)
])

prompt = prompt_test.partial(
        format = parser.get_format_instructions()
)
prompt

SyntaxError: incomplete input (1214419158.py, line 1)

### chain 실습
1. 프롬프트, 아웃파서 바꿔서 체인요청
2. chain 연결
3. OutputParser를 변경

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template(
""" 
너는 바다속에 사는 문어야
항상 헤헤헤 하고, 냥냥체를 사용하여 대답하자

질문:
{question}
""")

chain23 = prompt | model | StrOutputParser()



In [14]:
result = ({

    "question": "문어는 취미가 무엇이에요?"
})
print(result)

{'question': '문어는 취미가 무엇이에요?'}


In [16]:
character_prompt = """
  너는 바닷속에 사는 문어 캐릭터야.

  [성격]
  - 장난스럽고 친절해.
  - 어려운 내용도 쉽게 설명해.

  [말투]
  - 대답할 때 가끔 "헤헤헤"라고 말해.
  - 문장 끝에는 자연스럽게 "냥"을 붙여.
  - 너무 딱딱하게 말하지 마.

  [답변 규칙]
  - 질문에 정확하게 답해.
  - 모르는 내용은 모른다고 말해.
  - 3~5문장으로 답해.
  """

In [ ]:
chain = prompt | model
result = chain.invoke({"language" : "영어", 
              "question" : "주말에 할 일들을 추천해줘. 리스트형식으로 줘"})
print(result.text)

In [ ]:
from langchain_core.output_parsers import StrOutputParser, ListOutputParser

chain = prompt | model | StrOutputParser()
result = chain.invoke({"language" : "영어", 
              "question" : "나는 금요일이라서 너무 좋다. 주말에 공부를 더 할 수 있어서"})
print(result)

In [ ]:
# 프롬프트 바꿔보기
prompt = ChatPromptTemplate.from_messages([
    ('system', "{language}로 아래 내용에 답해주세요"),
    ('human', "{question}")
])

chain = prompt | model | CommaSeparatedListOutputParser()
result = chain.invoke({"language" : "영어", 
              "question" : "주말에 할 일들을 추천해줘. 콤마로 구분해서줘"})
print(result)

In [ ]:
# 프롬프트 바꿔보기
prompt = ChatPromptTemplate.from_messages([
    ('system', "{language}로 아래 내용에 답해주세요"),
    ('human', "{question}")
])

chain = prompt | model
result = chain.invoke({"language" : "영어", 
              "question" : "주말에 할 일들을 추천해줘. 콤마로 구분해서줘"})
print(result.text)

NameError: name 'character_prompt' is not defined

In [ ]:
연습하기 
영화취향을 입력받아 맟춪형 영화 3편 추천 LCEL 체인
사용자 기분에 맞는 음악을 추천하는 LCEL 체인
캐릭터 설정해서 캐릭터의 톤 말투에 맞춰 답변하는 LCEL 체인 만들기

In [ ]:
구조화된 출력
예전에는 pydantic 으로 강제했었다
model에서 구조화된 출력을 지원

In [25]:
from pydantic import BaseModel, Field

class Place(BaseModel):
    name : str = Field(description="추천 여행 장소명")
    reason : str = Field(description="추천한 이유")
    date : str = Field(description="추천 여행일")

structured_model = model.with_structured_output(Place, method="json_schema")
response = structured_model.invoke("주말에 갈만한 여행지 추천해줘")
print(response)

name='강릉' reason='서울·수도권에서 주말에 다녀오기 좋고, 안목해변 카페거리·경포호·초당순두부 등 바다와 맛집을 함께 즐길 수 있습니다. 여유로운 1박 2일 여행에 특히 추천합니다.' date='2026년 9월 19일~20일'


In [ ]:
print(response.name)

In [ ]:
result = response.model_dump() 
result

{'name': '강릉',
 'reason': '서울·수도권에서 주말에 다녀오기 좋고, 안목해변 카페거리·경포호·초당순두부 등 바다와 맛집을 함께 즐길 수 있습니다. 여유로운 1박 2일 여행에 특히 추천합니다.',
 'date': '2026년 9월 19일~20일'}

In [ ]:
#mmr 알고리즘 lambda_mult 1은 질문과 비슷한 문서 0은 서로다른 문서
retreiver = load_vs.as_retriever(
            search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25}
        )
#실제검색
result_mmr=retreiver.invoke("신혼 부부 지원 정책")
print(result_mmr)
